# 🧠⚡ NEURO BAND — Morning Brief System
### Your AI-powered daily dashboard: Flashcards • To-Do • Weather • Health

---
**What this notebook does:**
- 📚 **Flashcard Tracker** — Record & review yesterday's study cards
- ✅ **To-Do List** — Manage daily tasks with priority levels
- 🌤️ **Weather** — Live weather via `wttr.in` (100% FREE, no key needed)
- 🧬 **Health AI** — Personalized health tips via HuggingFace FREE API
- 🖥️ **Morning Brief** — Beautiful all-in-one dashboard

**Run cells in order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8**

In [ ]:
# ============================================================
# CELL 1 — INSTALL LIBRARIES
# Run this ONCE at the start of every Colab session
# ============================================================

!pip install requests ipywidgets -q

# Enable ipywidgets display in Colab
from google.colab import output
output.enable_custom_widget_manager()

print('✅ All libraries installed and ready!')
print('👉 Now run Cell 2 to configure your settings.')

In [ ]:
# ============================================================
# CELL 2 — CONFIGURATION
# Edit these 3 values before running
# ============================================================

# ✏️ YOUR SETTINGS — EDIT THESE:
YOUR_CITY      = "Colombo"          # e.g., "London", "Mumbai", "New York"
YOUR_NAME      = "Friend"           # Your name for the greeting
HF_TOKEN       = "hf_XXXXXXXXXXXXXXXXXXXXXXXX"  # From huggingface.co/settings/tokens (FREE)

# ─── HuggingFace FREE Model (best free text-gen model) ──────
HF_MODEL = "HuggingFaceH4/zephyr-7b-beta"  # Best free model — do NOT change
HF_API_URL = f"https://api-inference.huggingface.co/models/{HF_MODEL}"

# ─── Storage file (saved inside Colab /content/) ─────────────
import os
DATA_FILE = "/content/neuro_band_data.json"

print('✅ Configuration saved!')
print(f'   📍 City   : {YOUR_CITY}')
print(f'   👤 Name   : {YOUR_NAME}')
print(f'   🤖 Model  : {HF_MODEL}')
print()
print('📌 HOW TO GET A FREE HUGGINGFACE TOKEN:')
print('   1. Go to https://huggingface.co/join  → Create free account')
print('   2. Go to https://huggingface.co/settings/tokens')
print('   3. Click "New token" → Name it → Role: "Read" → Copy it')
print('   4. Paste it above replacing  hf_XXXXXXXXXXXXXXXXXXXXXXXX')
print()
print('👉 Now run Cell 3 to set up the data manager.')

In [ ]:
# ============================================================
# CELL 3 — DATA MANAGER (Runs silently in background)
# ============================================================

import json
import requests
from datetime import datetime, date, timedelta
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ─── Load / Save helpers ─────────────────────────────────────
def load_data():
    """Load all saved data from the JSON file."""
    default = {
        "flashcards": {},   # { "YYYY-MM-DD": [ {topic, question, answer, time}, ... ] }
        "todos": [],        # [ {task, priority, done, created, id}, ... ]
    }
    if os.path.exists(DATA_FILE):
        try:
            with open(DATA_FILE, 'r') as f:
                return json.load(f)
        except Exception:
            return default
    return default

def save_data(data):
    """Save all data to the JSON file."""
    with open(DATA_FILE, 'w') as f:
        json.dump(data, f, indent=2)

# ─── Initial load ─────────────────────────────────────────────
data = load_data()
today_str     = str(date.today())
yesterday_str = str(date.today() - timedelta(days=1))

print('✅ Data Manager Ready!')
print(f'   📂 Storage file : {DATA_FILE}')
print(f'   📚 Flashcard days on record : {len(data["flashcards"])}')
print(f'   📋 Total to-do items        : {len(data["todos"])}')
print()
print('👉 Now run Cell 4 — Flashcard Manager')

In [ ]:
# ============================================================
# CELL 4 — FLASHCARD MANAGER
# Add, view, and review your study flashcards
# ============================================================

def flashcard_manager():
    data = load_data()
    out  = widgets.Output()

    # ── Header ──────────────────────────────────────────────
    header = widgets.HTML(value="""
    <div style='background:linear-gradient(135deg,#667eea,#764ba2);
                padding:14px 18px; border-radius:10px; color:white; margin-bottom:12px;'>
        <h2 style='margin:0; font-family:sans-serif'>📚 Flashcard Manager</h2>
        <p  style='margin:4px 0 0; opacity:.75; font-family:sans-serif; font-size:.85em'>
            Add cards you studied today. They will appear in tomorrow's Morning Brief.
        </p>
    </div>
    """)

    # ── Input widgets ────────────────────────────────────────
    topic_in = widgets.Text(
        placeholder='Topic (e.g., Biology, Python)',
        layout=widgets.Layout(width='280px'))

    question_in = widgets.Textarea(
        placeholder='Question',
        layout=widgets.Layout(width='460px', height='60px'))

    answer_in = widgets.Textarea(
        placeholder='Answer',
        layout=widgets.Layout(width='460px', height='60px'))

    add_btn      = widgets.Button(description='➕ Add Card',         button_style='primary')
    view_btn     = widgets.Button(description='📖 View Today\'s',    button_style='info')
    view_yest_btn= widgets.Button(description='📅 View Yesterday\'s',button_style='warning')

    # ── Button callbacks ─────────────────────────────────────
    def add_card(b):
        data = load_data()
        today = str(date.today())
        q = question_in.value.strip()
        a = answer_in.value.strip()
        if not q or not a:
            with out:
                clear_output()
                print('⚠️  Please fill in both Question and Answer fields.')
            return
        if today not in data['flashcards']:
            data['flashcards'][today] = []
        data['flashcards'][today].append({
            'topic'    : topic_in.value.strip() or 'General',
            'question' : q,
            'answer'   : a,
            'time'     : datetime.now().strftime('%H:%M')
        })
        save_data(data)
        topic_in.value    = ''
        question_in.value = ''
        answer_in.value   = ''
        count = len(data['flashcards'][today])
        with out:
            clear_output()
            print(f'✅ Flashcard added! You now have {count} card(s) for today ({today}).')

    def view_today(b):
        data  = load_data()
        today = str(date.today())
        cards = data['flashcards'].get(today, [])
        with out:
            clear_output()
            if not cards:
                print("No flashcards added today yet. Use the form above to add some!")
                return
            print(f"📚 TODAY'S FLASHCARDS ({today}) — {len(cards)} total")
            print('='*50)
            for i, c in enumerate(cards, 1):
                print(f"\n  Card {i} | 🏷️ {c['topic']} | ⏰ {c['time']}")
                print(f"  ❓ Q: {c['question']}")
                print(f"  💡 A: {c['answer']}")

    def view_yesterday(b):
        data      = load_data()
        yesterday = str(date.today() - timedelta(days=1))
        cards     = data['flashcards'].get(yesterday, [])
        with out:
            clear_output()
            if not cards:
                print(f"No flashcards found for yesterday ({yesterday}).")
                return
            print(f"📅 YESTERDAY'S FLASHCARDS ({yesterday}) — {len(cards)} total")
            print('='*50)
            for i, c in enumerate(cards, 1):
                print(f"\n  Card {i} | 🏷️ {c['topic']} | ⏰ {c['time']}")
                print(f"  ❓ Q: {c['question']}")
                print(f"  💡 A: {c['answer']}")

    add_btn.on_click(add_card)
    view_btn.on_click(view_today)
    view_yest_btn.on_click(view_yesterday)

    display(
        header,
        widgets.HBox([topic_in]),
        question_in,
        answer_in,
        widgets.HBox([add_btn, view_btn, view_yest_btn],
                     layout=widgets.Layout(gap='8px', margin='8px 0')),
        out
    )

flashcard_manager()
print('\n👉 After adding cards, run Cell 5 — To-Do List')

In [ ]:
# ============================================================
# CELL 5 — TO-DO LIST MANAGER
# Add tasks, set priorities, mark them done
# ============================================================

def todo_manager():
    data = load_data()
    out  = widgets.Output()

    # ── Header ──────────────────────────────────────────────
    header = widgets.HTML(value="""
    <div style='background:linear-gradient(135deg,#11998e,#38ef7d);
                padding:14px 18px; border-radius:10px; color:white; margin-bottom:12px;'>
        <h2 style='margin:0; font-family:sans-serif'>✅ To-Do List Manager</h2>
        <p  style='margin:4px 0 0; opacity:.8; font-family:sans-serif; font-size:.85em'>
            Add, view, complete, and clear your daily tasks.
        </p>
    </div>
    """)

    # ── Input widgets ────────────────────────────────────────
    task_in = widgets.Text(
        placeholder='Enter a task...',
        layout=widgets.Layout(width='350px'))

    priority_drop = widgets.Dropdown(
        options=['🔴 High', '🟡 Medium', '🟢 Low'],
        value='🟡 Medium',
        layout=widgets.Layout(width='140px'))

    add_btn   = widgets.Button(description='➕ Add Task',     button_style='primary')
    view_btn  = widgets.Button(description='📋 View All',     button_style='info')

    done_num  = widgets.BoundedIntText(
        min=0, max=999, value=0,
        description='Task #:',
        layout=widgets.Layout(width='160px'))

    done_btn  = widgets.Button(description='✓ Mark Done',     button_style='success')
    clear_btn = widgets.Button(description='🗑️ Clear Done',   button_style='warning')

    # ── Callbacks ────────────────────────────────────────────
    def add_task(b):
        data = load_data()
        task = task_in.value.strip()
        if not task:
            with out:
                clear_output()
                print('⚠️  Task cannot be empty.')
            return
        new_id = max([t.get('id', 0) for t in data['todos']], default=0) + 1
        data['todos'].append({
            'id'      : new_id,
            'task'    : task,
            'priority': priority_drop.value,
            'done'    : False,
            'created' : str(date.today())
        })
        save_data(data)
        task_in.value = ''
        with out:
            clear_output()
            pending = sum(1 for t in data['todos'] if not t['done'])
            print(f'✅ Task added! You have {pending} pending task(s).')

    def view_tasks(b):
        data    = load_data()
        pending = [t for t in data['todos'] if not t['done']]
        done    = [t for t in data['todos'] if  t['done']]
        with out:
            clear_output()
            print(f"\n📋 PENDING TASKS ({len(pending)})")
            print('='*50)
            if not pending:
                print('  ✨ No pending tasks! You are all caught up.')
            else:
                for i, t in enumerate(pending):
                    print(f"  [{i}] {t['priority']}  |  {t['task']}  (added: {t['created']})")
            print(f"\n✅ COMPLETED TASKS ({len(done)})")
            print('='*50)
            if not done:
                print('  None completed yet.')
            else:
                for t in done:
                    print(f"  ✓  {t['task']}")

    def mark_done(b):
        data    = load_data()
        pending = [t for t in data['todos'] if not t['done']]
        idx     = done_num.value
        if not pending:
            with out:
                clear_output()
                print('ℹ️  No pending tasks to complete.')
            return
        if idx < 0 or idx >= len(pending):
            with out:
                clear_output()
                print(f'⚠️  Invalid task number. Enter 0–{len(pending)-1}.')
                print('Tip: Click "📋 View All" first to see task numbers.')
            return
        # Mark it done in the master list
        task_name = pending[idx]['task']
        target_id = pending[idx]['id']
        for t in data['todos']:
            if t['id'] == target_id:
                t['done'] = True
                break
        save_data(data)
        with out:
            clear_output()
            print(f'✅  "{task_name}" marked as done! Great work!')

    def clear_done(b):
        data = load_data()
        removed = sum(1 for t in data['todos'] if t['done'])
        data['todos'] = [t for t in data['todos'] if not t['done']]
        save_data(data)
        with out:
            clear_output()
            print(f'🗑️  Cleared {removed} completed task(s). Clean slate!')

    add_btn.on_click(add_task)
    view_btn.on_click(view_tasks)
    done_btn.on_click(mark_done)
    clear_btn.on_click(clear_done)

    display(
        header,
        widgets.HBox([task_in, priority_drop, add_btn],
                     layout=widgets.Layout(gap='6px', align_items='center')),
        widgets.HBox([view_btn, clear_btn],
                     layout=widgets.Layout(gap='6px', margin='6px 0')),
        widgets.HTML(value="<p style='font-family:sans-serif; font-size:.85em; margin:4px 0; color:#555;'>"
                           "To mark a task done: click 📋 View All to see numbers, then enter the number below.</p>"),
        widgets.HBox([done_num, done_btn],
                     layout=widgets.Layout(gap='6px', align_items='center')),
        out
    )

todo_manager()
print('\n👉 After setting up your tasks, run Cell 6 — Weather')

In [ ]:
# ============================================================
# CELL 6 — LIVE WEATHER  (wttr.in — 100% FREE, NO API KEY)
# ============================================================

def get_weather(city=YOUR_CITY):
    """Fetch live weather from wttr.in (completely free, no API key required)."""
    print(f'🌐 Fetching weather for {city}...')
    try:
        url  = f'https://wttr.in/{city.replace(" ","+")}?format=j1'
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        raw  = resp.json()

        cur  = raw['current_condition'][0]
        tod  = raw['weather'][0]

        info = {
            'city'       : city,
            'desc'       : cur['weatherDesc'][0]['value'],
            'temp_c'     : cur['temp_C'],
            'feels_c'    : cur['FeelsLikeC'],
            'humidity'   : cur['humidity'],
            'wind_kmph'  : cur['windspeedKmph'],
            'uv_index'   : cur['uvIndex'],
            'visibility' : cur['visibility'],
            'min_c'      : tod['mintempC'],
            'max_c'      : tod['maxtempC'],
        }

        print()
        print('🌤️  LIVE WEATHER REPORT')
        print('='*44)
        print(f"  📍 City        : {info['city']}")
        print(f"  🌡️  Temperature : {info['temp_c']}°C  (feels like {info['feels_c']}°C)")
        print(f"  ☁️  Condition   : {info['desc']}")
        print(f"  💧 Humidity    : {info['humidity']}%")
        print(f"  💨 Wind        : {info['wind_kmph']} km/h")
        print(f"  ☀️  UV Index    : {info['uv_index']}")
        print(f"  👁️  Visibility  : {info['visibility']} km")
        print(f"  📊 Today Range : {info['min_c']}°C — {info['max_c']}°C")
        print()
        print('✅ Weather fetched successfully!')
        return info

    except requests.exceptions.ConnectionError:
        print('❌ No internet connection. Check your network.')
    except requests.exceptions.Timeout:
        print('❌ Request timed out. Try again in a moment.')
    except Exception as e:
        print(f'❌ Weather fetch error: {e}')

    # Fallback so later cells don't crash
    return {
        'city':'Unknown','desc':'Unavailable','temp_c':'N/A','feels_c':'N/A',
        'humidity':'N/A','wind_kmph':'N/A','uv_index':'0',
        'visibility':'N/A','min_c':'N/A','max_c':'N/A'
    }


# Run it
weather_info = get_weather(YOUR_CITY)
print('\n👉 Now run Cell 7 — AI Health Recommendations')

In [ ]:
# ============================================================
# CELL 7 — AI HEALTH RECOMMENDATIONS (HuggingFace FREE API)
# Model: HuggingFaceH4/zephyr-7b-beta
# ============================================================

import time

def get_health_recommendations(weather, todos, yesterday_cards):
    """Call HuggingFace Inference API for personalised health tips."""

    pending_count  = sum(1 for t in todos if not t['done'])
    card_count     = len(yesterday_cards)
    topics_studied = list(set(c.get('topic', 'General') for c in yesterday_cards)) if yesterday_cards else []

    prompt = f"""<|system|>
You are a concise health & wellness coach. Give exactly 5 practical health tips.
Each tip must be one sentence. Number them 1 to 5. No extra text.<|endoftext|>
<|user|>
My morning data:
- Location: {weather.get('city','Unknown')}
- Weather: {weather.get('desc','Unknown')}, {weather.get('temp_c','?')}°C
- Humidity: {weather.get('humidity','?')}%
- UV Index: {weather.get('uv_index','?')}
- Wind: {weather.get('wind_kmph','?')} km/h
- Pending tasks today: {pending_count}
- Flashcards studied yesterday: {card_count} ({', '.join(topics_studied) if topics_studied else 'none'})

Give me 5 specific health tips for today based on this data.<|endoftext|>
<|assistant|>"""

    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 350,
            "temperature": 0.7,
            "do_sample": True,
            "return_full_text": False,
            "stop": ["<|endoftext|>", "<|user|>"]
        }
    }

    print('🤖 Calling HuggingFace AI... (may take 15–45 seconds on first call)')
    print('   Model is loading if this is your first call today — please wait.')

    for attempt in range(3):  # Retry up to 3 times (model may be loading)
        try:
            resp = requests.post(HF_API_URL, headers=headers, json=payload, timeout=90)

            if resp.status_code == 503:  # Model loading
                wait = resp.json().get('estimated_time', 20)
                print(f'   ⏳ Model loading... waiting {int(wait)+5}s (attempt {attempt+1}/3)')
                time.sleep(int(wait) + 5)
                continue

            if resp.status_code == 401:
                print('❌ Invalid HuggingFace token. Check Cell 2 and re-run.')
                return None

            resp.raise_for_status()
            result = resp.json()

            if isinstance(result, list) and result:
                text = result[0].get('generated_text', '').strip()
                print()
                print('🧬 AI HEALTH RECOMMENDATIONS FOR TODAY')
                print('='*44)
                print(text)
                print()
                print('✅ AI recommendations generated!')
                return text
            else:
                raise ValueError(f'Unexpected API response: {result}')

        except requests.exceptions.Timeout:
            print(f'   ⏳ Timeout on attempt {attempt+1}. Retrying...')
            time.sleep(10)
        except Exception as e:
            print(f'   ⚠️  Attempt {attempt+1} failed: {e}')
            time.sleep(5)

    # ── Fallback: rule-based tips if API unavailable ─────────
    print()
    print('⚠️  HuggingFace API unavailable. Using smart fallback recommendations.')
    print()
    tips = []
    uv   = int(weather.get('uv_index', 0)) if str(weather.get('uv_index','0')).isdigit() else 0
    hum  = int(weather.get('humidity', 50)) if str(weather.get('humidity','50')).isdigit() else 50
    temp = int(weather.get('temp_c', 25)) if str(weather.get('temp_c','25')).lstrip('-').isdigit() else 25
    wind = int(weather.get('wind_kmph', 0)) if str(weather.get('wind_kmph','0')).isdigit() else 0

    if uv >= 8:
        tips.append('1. ☀️ UV index is very high — apply SPF 50+ sunscreen and wear a hat outdoors.')
    elif uv >= 5:
        tips.append('1. 🧴 UV index is moderate-high — use SPF 30 sunscreen if going outside.')
    else:
        tips.append('1. 🚶 UV index is low — great day for an outdoor walk or light exercise.')

    if hum > 75:
        tips.append('2. 💧 High humidity today — drink an extra 500ml of water and stay in shade.')
    elif hum < 35:
        tips.append('2. 💧 Low humidity — moisturise your skin and drink water every 45 minutes.')
    else:
        tips.append('2. 💧 Humidity is comfortable — aim for 8 glasses of water throughout the day.')

    if temp > 32:
        tips.append('3. 🧊 It is very hot — avoid outdoor activity between 11am–3pm; eat light meals.')
    elif temp < 18:
        tips.append('3. 🧥 It is cool today — layer up before heading out to protect your joints.')
    else:
        tips.append('3. 🌿 Great temperature for a 20-minute morning walk to boost your metabolism.')

    if pending_count > 5:
        tips.append('4. 🧠 You have many tasks — use the Pomodoro technique: 25 min work, 5 min break.')
    else:
        tips.append('4. 🧠 Follow the 20-20-20 rule: every 20 minutes look 20 feet away for 20 seconds.')

    if card_count > 0:
        tips.append(f'5. 📚 You studied {card_count} flashcard(s) yesterday — quick review before bed locks in memory.')
    else:
        tips.append('5. 📚 No flashcards yesterday — start small: add 5 cards today to build a study habit.')

    result_text = '\n'.join(tips)
    print('🧬 SMART HEALTH RECOMMENDATIONS FOR TODAY')
    print('='*44)
    print(result_text)
    return result_text


# ── Run it ───────────────────────────────────────────────────
data_now       = load_data()
yesterday_cards = data_now['flashcards'].get(yesterday_str, [])

health_recs = get_health_recommendations(weather_info, data_now['todos'], yesterday_cards)
print('\n👉 Now run Cell 8 — Your Morning Brief Dashboard!')

In [ ]:
# ============================================================
# CELL 8 — 🧠 MORNING BRIEF DASHBOARD  (Full Visual)
# ============================================================

def generate_morning_brief():
    data       = load_data()
    today_date = datetime.now().strftime('%A, %B %d, %Y')
    today_time = datetime.now().strftime('%I:%M %p')

    # ── Data ─────────────────────────────────────────────────
    yest_cards  = data['flashcards'].get(yesterday_str, [])
    today_cards = data['flashcards'].get(today_str,     [])
    pending     = [t for t in data['todos'] if not t['done']]
    done_tasks  = [t for t in data['todos'] if  t['done']]
    yest_topics = list(set(c.get('topic','General') for c in yest_cards))

    # ── Weather block ─────────────────────────────────────────
    w = weather_info
    weather_html = f"""
    <div style='display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin:10px 0;'>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>🌡️</div>
        <div style='font-size:1.1em;font-weight:700'>{w['temp_c']}°C</div>
        <div style='opacity:.6;font-size:.75em'>Temp (feels {w['feels_c']}°C)</div>
      </div>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>💧</div>
        <div style='font-size:1.1em;font-weight:700'>{w['humidity']}%</div>
        <div style='opacity:.6;font-size:.75em'>Humidity</div>
      </div>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>☀️</div>
        <div style='font-size:1.1em;font-weight:700'>{w['uv_index']}</div>
        <div style='opacity:.6;font-size:.75em'>UV Index</div>
      </div>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>💨</div>
        <div style='font-size:1.1em;font-weight:700'>{w['wind_kmph']}</div>
        <div style='opacity:.6;font-size:.75em'>Wind km/h</div>
      </div>
    </div>
    <div style='text-align:center;opacity:.7;font-size:.85em;'>
      {w['desc']} &nbsp;|&nbsp; {w['min_c']}°C – {w['max_c']}°C today
    </div>"""

    # ── Yesterday's flashcards block ─────────────────────────
    if yest_cards:
        card_items = ''.join([
            f"<li style='margin:5px 0'>"
            f"<span style='background:rgba(102,126,234,.5);padding:1px 7px;border-radius:10px;"
            f"font-size:.75em;'>{c.get('topic','?')}</span> "
            f"{c['question'][:55]}{'…' if len(c['question'])>55 else ''}</li>"
            for c in yest_cards[:6]
        ])
        extra = f"<li style='opacity:.5'>…and {len(yest_cards)-6} more</li>" if len(yest_cards)>6 else ""
        fc_html = f"<ul style='margin:6px 0;padding-left:18px;'>{card_items}{extra}</ul>"
    else:
        fc_html = "<p style='opacity:.5;margin:6px 0;'>No flashcards recorded yesterday. Add some in Cell 4!</p>"

    # ── To-do block ──────────────────────────────────────────
    if pending:
        priority_order = {'🔴 High': 0, '🟡 Medium': 1, '🟢 Low': 2}
        sorted_pending = sorted(pending, key=lambda t: priority_order.get(t['priority'], 1))
        badge_colors   = {'🔴 High':'#ff4757','🟡 Medium':'#ffa502','🟢 Low':'#2ed573'}
        todo_items = ''.join([
            f"<li style='margin:5px 0;'>"
            f"<span style='background:{badge_colors.get(t['priority'],'#888')};"
            f"padding:1px 7px;border-radius:10px;font-size:.75em;color:#fff;'>"
            f"{t['priority'].split(' ')[-1].upper()}</span> {t['task']}</li>"
            for t in sorted_pending[:8]
        ])
        todo_html = f"<ul style='margin:6px 0;padding-left:18px;'>{todo_items}</ul>"
    else:
        todo_html = "<p style='opacity:.5;margin:6px 0;'>✨ No pending tasks! You are all caught up.</p>"

    # ── Health recs block ────────────────────────────────────
    health_text = str(health_recs).replace('\n','<br>') if health_recs else \
        'Run Cell 7 to generate AI health recommendations.'

    # ── Assemble full HTML ────────────────────────────────────
    html = f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;800&family=DM+Sans:wght@300;400;500&display=swap');
      .nb * {{ font-family: 'DM Sans', sans-serif; color: white; box-sizing: border-box; }}
      .nb h1, .nb h2, .nb h3 {{ font-family: 'Syne', sans-serif; }}
      .nb-section {{
        background: rgba(255,255,255,.06);
        border-radius: 14px;
        padding: 14px 16px;
        margin-bottom: 14px;
      }}
      .nb-label {{
        font-size: .75em;
        letter-spacing: 2px;
        opacity: .55;
        font-family: 'Syne', sans-serif;
        text-transform: uppercase;
        margin-bottom: 6px;
      }}
    </style>

    <div class='nb' style='
        background: linear-gradient(145deg,#0f0c29 0%,#302b63 55%,#1a1a2e 100%);
        padding: 28px 24px;
        border-radius: 22px;
        max-width: 720px;
        border: 1px solid rgba(255,255,255,.08);
        box-shadow: 0 30px 80px rgba(0,0,0,.6);
    '>

      <!-- ═══ HEADER ═════════════════════════════════════ -->
      <div style='text-align:center;margin-bottom:22px;
                  padding-bottom:18px;border-bottom:1px solid rgba(255,255,255,.1);'>
        <div style='font-size:2.8em;margin-bottom:4px;'>🧠⚡</div>
        <h1 style='margin:0;font-size:2em;font-weight:800;letter-spacing:3px;'>NEURO BAND</h1>
        <p style='margin:2px 0 0;opacity:.45;font-size:.8em;letter-spacing:4px;font-family:Syne,sans-serif;'>MORNING BRIEF SYSTEM</p>
        <div style='margin-top:12px;display:flex;justify-content:center;gap:24px;opacity:.7;font-size:.85em;'>
          <span>👋 Good morning, {YOUR_NAME}!</span>
          <span>📅 {today_date}</span>
          <span>⏰ {today_time}</span>
        </div>
        <div style='margin-top:10px;display:flex;justify-content:center;gap:16px;'>
          <span style='background:rgba(102,126,234,.3);padding:3px 12px;border-radius:20px;font-size:.8em;'>
            📚 {len(yest_cards)} cards yesterday</span>
          <span style='background:rgba(17,153,142,.3);padding:3px 12px;border-radius:20px;font-size:.8em;'>
            ✅ {len(pending)} tasks pending</span>
          <span style='background:rgba(255,71,87,.3);padding:3px 12px;border-radius:20px;font-size:.8em;'>
            🏆 {len(done_tasks)} tasks done</span>
        </div>
      </div>

      <!-- ═══ WEATHER ═══════════════════════════════════ -->
      <div class='nb-section'>
        <div class='nb-label'>🌤️ Weather — {w['city']}</div>
        {weather_html}
      </div>

      <!-- ═══ FLASHCARDS ════════════════════════════════ -->
      <div class='nb-section'>
        <div class='nb-label'>📚 Yesterday's Flashcards — {len(yest_cards)} cards | Topics: {', '.join(yest_topics) if yest_topics else 'None'}</div>
        {fc_html}
      </div>

      <!-- ═══ TO-DO ══════════════════════════════════════ -->
      <div class='nb-section'>
        <div class='nb-label'>✅ To-Do List — {len(pending)} pending / {len(done_tasks)} done</div>
        {todo_html}
      </div>

      <!-- ═══ HEALTH AI ════════════════════════════════ -->
      <div style='
        background: rgba(102,126,234,.15);
        border: 1px solid rgba(102,126,234,.35);
        border-radius: 14px;
        padding: 14px 16px;
        margin-bottom: 14px;
      '>
        <div class='nb-label'>🧬 AI Health Brief — Powered by HuggingFace</div>
        <div style='line-height:1.75;font-size:.88em;opacity:.9;'>{health_text}</div>
      </div>

      <!-- ═══ FOOTER ════════════════════════════════════ -->
      <div style='text-align:center;opacity:.3;font-size:.7em;letter-spacing:2px;margin-top:4px;'>
        NEURO BAND MORNING BRIEF &nbsp;•&nbsp; wttr.in &nbsp;+&nbsp; HuggingFace &nbsp;•&nbsp; {today_date}
      </div>

    </div>"""

    display(HTML(html))


# ── Generate the brief ────────────────────────────────────────
generate_morning_brief()

---
## 📌 Daily Usage Guide

| When | What to do |
|------|------------|
| **Every morning** | Run Cell 1 (install) → Cell 3 (data) → Cell 6 (weather) → Cell 7 (AI) → Cell 8 (brief) |
| **During study sessions** | Run Cell 4 to add flashcards as you study |
| **Planning your day** | Run Cell 5 to add/manage tasks |
| **End of day** | Run Cell 5 → mark tasks done → run Cell 8 to see updated brief |

### ⚡ Quick Re-run (after first setup)
Every new Colab session, just run **Cells 1 → 3 → 6 → 7 → 8** to get your Morning Brief instantly.

---
**Data persists** in `/content/neuro_band_data.json` during a session.  
**Tip:** Download this file to your Google Drive before closing Colab to save your data permanently.